# DistilBERT Basic Test

This notebook runs a minimal sentiment inference test using the pretrained DistilBERT model `distilbert-base-uncased-finetuned-sst-2-english`.

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

print(f"Loaded model: {MODEL_NAME}")
print(f"Labels: {model.config.id2label}")

/home/tc/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3019.53it/s]

Loaded model: distilbert-base-uncased-finetuned-sst-2-english
Labels: {0: 'NEGATIVE', 1: 'POSITIVE'}


In [4]:
samples = [
    "Please review the attached budget report by end of day.",
    "Great work on the release, the rollout went smoothly!",
    "I am disappointed with the repeated delay in responses."
]

inputs = tokenizer(samples, return_tensors="pt", padding=True, truncation=True)

with torch.no_grad():
    logits = model(**inputs).logits

probs = torch.softmax(logits, dim=-1)
preds = torch.argmax(probs, dim=-1)

for text, pred, prob in zip(samples, preds, probs):
    label = model.config.id2label[pred.item()]
    confidence = prob[pred].item()
    print(f"Text: {text}")
    print(f"Prediction: {label} ({confidence:.3f})")
    print("-" * 70)

Text: Please review the attached budget report by end of day.
Prediction: NEGATIVE (0.968)
----------------------------------------------------------------------
Text: Great work on the release, the rollout went smoothly!
Prediction: POSITIVE (1.000)
----------------------------------------------------------------------
Text: I am disappointed with the repeated delay in responses.
Prediction: NEGATIVE (1.000)
----------------------------------------------------------------------


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import re

SUMMARIZER_MODEL = "google/flan-t5-small"
summary_tokenizer = AutoTokenizer.from_pretrained(SUMMARIZER_MODEL)
summary_model = AutoModelForSeq2SeqLM.from_pretrained(SUMMARIZER_MODEL)

STOPWORDS = {
    "a", "an", "the", "i", "me", "my", "we", "our", "you", "your", "he", "she", "it", "they", "them",
    "is", "am", "are", "was", "were", "be", "been", "being", "to", "of", "in", "on", "for", "with"
}

def clean_words(text: str) -> list[str]:
    tokens = [w.lower() for w in re.findall(r"[A-Za-z]+", text)]
    return [w for w in tokens if w not in STOPWORDS and len(w) > 1]

def summarize_1_2_words(text: str) -> str:
    prompt = (
        #"Write a 1 word topic label for this message. "
        "Provide a 1 word category describing the topic of this message"
        "Use content words only, no pronouns: "
        f"{text}"
    )
    inputs = summary_tokenizer(prompt, return_tensors="pt", truncation=True)

    output_ids = summary_model.generate(
        **inputs,
        max_new_tokens=3,
        num_beams=4,
        #do_sample=False,
        do_sample = True,
        temperature=0.5,
        top_p=0.9,
    )

    result = summary_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    words = clean_words(result)
    if not words:
        words = clean_words(text)
    if not words:
        return "No Summary"
    return " ".join(words[:1]).title()

summary_samples = [
    "Please review the attached budget report by end of day.",
    "Great work on the release, the rollout went smoothly!",
    "I am disappointed with the repeated delay in responses."
]

for s in summary_samples:
    print(f"Text: {s}")
    print(f"1-2 word summary: {summarize_1_2_words(s)}")
    print("-" * 70)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 3705.78it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Text: Please review the attached budget report by end of day.
1-2 word summary: Budget
----------------------------------------------------------------------
Text: Great work on the release, the rollout went smoothly!
1-2 word summary: Release
----------------------------------------------------------------------
Text: I am disappointed with the repeated delay in responses.
1-2 word summary: Response
----------------------------------------------------------------------
